# Homework 1: Basics of Language Modelling

This homework focuses on your formal and conceptual understanding of statistical learning, neural networks and foundations of language modelling. Follow along and try to answer the exercises on your own.

## Logistics TBD

## Part 1 — Probability over sequences

A language model is a probability distribution over sequences of tokens, and the earliest language models were $n$-grams — Markov chains of order $n-1$ 
estimated from a corpus. 

Consider the following corpus $\mathcal{C}_1$, sampled 
from a probabilistic context-free grammar $G$, lowercased and stripped of punctuation:

$$\mathcal{C}_1 = \begin{cases} \texttt{the dog saw the cat} \\ \texttt{the cat saw the bird} \\ \texttt{the bird saw the worm} \\ \texttt{the worm did not see the dog} \\ \texttt{the dog ate the worm} \\ \texttt{the cat ate the bird} \end{cases}$$


**1.1** A vocabulary is the set of distinct symbols a model can assign probability to. Define two vocabularies over $\mathcal{C}$: $\mathcal{V}_a$, the set of unique characters (treat space as a character), and $\mathcal{V}_b$, the set of unique space-delimited tokens. List the members of each and compute $|\mathcal{V}_a|$ and $|\mathcal{V}_b|$. For a corpus ten times larger drawn from $G$, which vocabulary would grow and which would not? Why?


**1.2** A unigram model assigns each symbol a probability proportional to how often it appears, ignoring all context. For each vocabulary, compute the unigram probability $p(w) = c(w)/N$ for every $w$. Verify that each distribution sums to 1. Now compute the probability the unigram model assigns to the sequence `the dog saw` under $\mathcal{V}_b$, and to the character sequence `t h e` (with intervening spaces) under $\mathcal{V}_a$. What does the unigram model fail to capture that is plainly visible in the corpus?


**1.3** A trigram model conditions each token on the two preceding tokens, estimating $p(w_t \mid w_{t-2}, w_{t-1})$ from counts. Working with $\mathcal{V}_b$, list all trigrams in $\mathcal{C}$ and their counts. Estimate $p(\textit{cat} \mid \textit{the, dog})$, $p(\textit{worm} \mid \textit{the, dog})$, and $p(\textit{bird} \mid \textit{the, dog})$. What does the model assign to $p(\textit{fish} \mid \textit{the, dog})$, and what does that imply about the model's behaviour on text that differs even slightly from $\mathcal{C}$?


**1.4** For a vocabulary of size $|\mathcal{V}|$, a unigram model requires $|\mathcal{V}| - 1$ free parameters (the last is determined by the sum-to-one constraint). An exact trigram model requires at most $|\mathcal{V}|^2 \cdot (|\mathcal{V}| - 1)$ free parameters. Compute these counts for both $\mathcal{V}_a$ and $\mathcal{V}_b$. For a real English corpus with $|\mathcal{V}_b| \approx 50{,}000$, how many parameters does the trigram table require? What does this suggest about the feasibility of exact $n$-gram models?

## Part 2 — The training objective

#### *The logarithm of likelihood:*
The likelihood of the corpus is the probability the model assigns to 
it, viewed as a function of its parameters $\theta$. Concretely, it answers: 
given this model, how probable is what we observed? Under a unigram model, 
the corpus tokens are independent, so this probability is simply the product 
of the individual token probabilities. In other words, a unigram model 
is parametrised by $\theta = \{p(w) : w \in \mathcal{V}\}$. 
A higher likelihood means the model is a better fit to the data.

Suppose your entire corpus is made up of the three tokens $\mathcal{C_2}=\{\text{the}, \text{sat}, \text{sat}\}$.
Then the likelihood of this corpus under a unigram model is

$$\mathcal{L}(\theta) = p(\textit{the}) \cdot p(\textit{sat}) \cdot p(\textit{sat})$$

Most often the likelihood is a product of many small numbers. So we work with 
its logarithm instead — the log-likelihood — which turns the product into 
a sum without changing which $\theta$ maximises it:

$$\log \mathcal{L}(\theta) = \sum_{t=1}^{N} \log p_\theta(w_t)$$

Maximum likelihood estimation finds the $\theta$ that makes the observed 
corpus as probable as possible. Since the log-likelihood is a smooth function 
of $\theta$, the optimal parameters are found where its derivative is zero.

**2.1** For the corpus $C_1$, 
- a. Show that the optimal parameters of the unigram model
are $\hat{p}(w) = c(w)/N$ for each word. 
>*hint: $p(\textit{sat}) = 1 - p(\textit{the})$*
- b. How does this hold for a corpus with a larger number of vocabulary?


#### *Comparing two distributions:*

Training a model means adjusting its parameters $\theta$ to reduce the loss. 
In the unigram model, $\theta = \{p(w) : w \in \mathcal{V}\}$ — the parameters 
are the probabilities themselves, and the context window is zero: each token 
is predicted without looking at any preceding tokens. Moving to a bigram model 
extends the context window to one: $\theta = \{p(w \mid w') : w, w' \in \mathcal{V}\}$, 
and to a trigram, to two. In each case the parameters remain probabilities 
directly — only the conditioning context grows. A neural language model keeps 
the fixed context window but replaces the direct parameterisation with a 
learned function: $\theta$ is now a set of weight matrices, and $p_\theta(w)$ 
is the result of passing the sequences of tokens through several layers of computation. 

Neural language models are often trained with the objective of minimising cross-entropy loss.
Cross-entropy measures how well one distribution predicts samples drawn from 
another. Given the true distribution $q$ and a model distribution $p$, it 
asks: on average, how many bits does $p$ need to encode a symbol drawn from 
$q$? If $p$ matches $q$ perfectly, cross-entropy equals the entropy of $q$ — 
the theoretical minimum. The further $p$ is from $q$, the higher the cross-entropy.

$$H(q, p) = -\sum_{w} q(w) \log p(w)$$

**2.2** Given a paramterised model $p_{\theta}$, 
- a. Show that the cross-entropy $H(\hat{p},\, p_\theta)$ is equal to the 
negative average log-likelihood of the corpus under $p_\theta$. 
- b. Show that maximising the log-likelihood is equivalent to minimising the cross-entropy 
of $p_\theta$ relative to the empirical distribution $\hat{p}$.

## Part 3 — Gradients

N-gram models estimate their parameters directly from data by counting and 
normalising. A neural language model cannot do this: its parameters are weight 
matrices with no direct correspondence to probabilities, and must instead be 
optimised iteratively. At each step, we ask: in which direction should $\theta$ 
move to reduce the loss? The gradient answers this — it is the vector of 
partial derivatives of the loss with respect to each parameter, pointing in 
the direction of steepest increase. Backpropagation is the algorithm that 
computes it.

A *deep* neural network computes its output by passing data through a sequence of 
layers, each a function of the previous one. The loss is therefore a 
composition of functions, and the gradient of the loss with respect to 
parameters in any given layer must account for every layer that follows. 
This is precisely what the chain rule handles.

The chain rule states that if $L$ depends on $z$ through $\hat{y}$, then

$$\frac{\partial L}{\partial z_i} = \sum_j \frac{\partial L}{\partial \hat{y}_j} \cdot \frac{\partial \hat{y}_j}{\partial z_i}$$

This decomposes the problem into two parts: how does the loss change with 
the probabilities, and how do the probabilities change with the logits.

The final layer of a neural language model produces a vector of real-valued 
scores — one per token in the vocabulary — called logits. These are 
unconstrained: they can be negative or arbitrarily large. To interpret them 
as probabilities, we need values that are non-negative and sum to one. 
The softmax converts logits into exactly such a distribution, and the 
cross-entropy loss is then applied to the result.

**3.1** Consider the computation $z \xrightarrow{\text{softmax}} \hat{y} \xrightarrow{\text{cross-entropy}} L$, where

$$\hat{y}_i = \frac{\exp(z_i)}{\sum_j \exp(z_j)}, \qquad L = -\sum_i y_i \log \hat{y}_i$$

and $y$ is a one-hot target vector.

- a. Compute $\partial L / \partial \hat{y}_j$. 

- b. Compute $\partial \hat{y}_j / \partial z_i$ for two cases: $i = j$ and $i \neq j$. 
Show that in both cases the result is $\hat{y}_j(\delta_{ij} - \hat{y}_i)$. 
The full matrix of these partial derivatives — one for each pair $(i, j)$ — 
is called the Jacobian of $\hat{y}$ with respect to $z$.

- c. Combine parts a and b via the chain rule to compute $\partial L / \partial z_i$. 
Show that the result simplifies to $\hat{y}_i - y_i$.